# 2022-11-04

현재의 setting으로는 학습을 하지 못함 & 기존의 그래프와 많은 차이가 나타남  
![](https://i.imgur.com/p4RgAoU.png)  

hparams tuning을 통해서 학습이 제대로 되도록 변경   


## sparsity

기존의 시간적인 sparsity를 해결하기 위해서 ewma를 적용했다.  
pandas의 moving average 여러 방식을 적용해서 ewma를 적용 후 학습 결과 확인  
현재 다른 variable이 noise처럼 작용할 수 있으므로 trainer를 만들 때 `count`와 `h_dong`만을 사용해서 학습을 진행(데이터의 연관성을 확인하지 않아서 데이터가 부정확할 수 있음)   

#### 차이점  
* time_idx 계산법 변경  
* MOY(Month of Year)추가  
* epoch : 200 -> 50 으로 감소  
* hidden_size : 16 -> 32로 증가  

#### Trainer
```
imeSeriesDataSet[length=183939](
	time_idx='time_idx',
	target='count',
	group_ids=['h_dong'],
	weight=None,
	max_encoder_length=168,
	min_encoder_length=84,
	min_prediction_idx=0,
	min_prediction_length=1,
	max_prediction_length=24,
	static_categoricals=['h_dong'],
	static_reals=['encoder_length'],
	time_varying_known_categoricals=[],
	time_varying_known_reals=['relative_time_idx'],
	time_varying_unknown_categoricals=[],
	time_varying_unknown_reals=['count'],
	variable_groups={},
	constant_fill_strategy={},
	allow_missing_timesteps=True,
	lags={},
	add_relative_time_idx=True,
	add_target_scales=False,
	add_encoder_length=True,
	target_normalizer=GroupNormalizer(
	method='standard',
	groups=['h_dong'],
	center=False,
	scale_by_group=False,
	transformation='relu',
	method_kwargs={}
),
	categorical_encoders={'__group_id__h_dong': NaNLabelEncoder(add_nan=False, warn=True), 'h_dong': NaNLabelEncoder(add_nan=False, warn=True)},
	scalers={'encoder_length': StandardScaler(), 'relative_time_idx': StandardScaler()},
	randomize_length=None,
	predict_mode=False
)
```

#### result

##### iter1
> result_notebook_iter1 : http://202.31.200.194:8888/notebooks/NPLAB-NAS/Members/SEO/Emergency_Demand/TFT/TEST_ewma/result_compare_iter1.ipynb


![](https://i.imgur.com/33AUcw4.png)

다음과 같은 그래프가 나타남 -> 그래도 아직은 학습을 제대로 하지 못하고 있음  

##### iter2
ewma를 중복 적용후 다시 확인

> result_notebook_iter2 : http://202.31.200.194:8888/notebooks/NPLAB-NAS/Members/SEO/Emergency_Demand/TFT/TEST_ewma/result_compare_%20iter2.ipynb


![](https://i.imgur.com/FXYOvGR.png)

그래도 여전히 이상한 모습임  



# lr_find

pytorch_forecasting의 tutorial에서는 lr_finder를 이용해서 lr를 결정함  
현재 lr = 0.03(tutorial에서의 setting)으로 해놓고 학습 -> 데이터가 희소하므로 lr을 낮춰서 학습  

lr_finder를 이용해서 학습을 진행  

> error : http://202.31.200.194:8888/notebooks/NPLAB-NAS/Members/SEO/Emergency_Demand/TFT/TEST_ewma/training_GPU0_lr_tuner.ipynb


# lr_hand
우선 수동으로 lr_tuning 진행  
http://202.31.200.194:8888/notebooks/NPLAB-NAS/Members/SEO/Emergency_Demand/TFT/TEST_ewma/training_GPU0_lr_hand_ewma2.ipynb


lr = 1 일 때  예측을 잘하고 있음  

![](https://i.imgur.com/Y3g40UY.png)

lr : 1e-5 ~ 1e+5 까지해서 학습  
lr을 1 이상의 값으로 함으로써 일종의 sclae up 효과 발생?  
